In [ ]:
# Purpose - plots cutaways for the big boy (141)

findex = 140
%load_ext autoreload
%autoreload

steps = range(100, 212, 1) # TODO get good range

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
from fmgmt.TristanRun import TristanRun
from fmgmt.FieldLineHelper import FieldLineHelper

plt.close('all')
plt.ioff() 

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps}

boilerplate = "NOT FILLED IN"

In [ ]:
# Configure Data

d = run.config_data(steps, "fields", includes_pressure = False)

In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim

#rect = [xlim / 2 - 4, xlim / 2 + 4, ylim / 2 - 10, ylim / 2 + 10]
#rect = [0, xlim, 0, ylim]
#rect = [50, 70, 130, 190]
rect = [0, xlim, 0, ylim]
rect = [5, xlim - 5, ylim / 2 - 240, ylim / 2 + 240]

tchosen = 0
roi = fq_cut(d.fields, tchosen, rect)

#xcut_y = 164.4
xcut_y = ylim / 10 #5950
ycut_x = xlim / 2

xstrip = fq_xcut(roi, xcut_y)
ystrip = fq_ycut(roi, ycut_x)

In [ ]:
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim])

#roi = fq_cut(d.fields, 0, [55, 65, 110, 130])

# Create a subplot with one plot
plt.figure()
fig, ax = plt.subplots()

#xa, ya = [0.02, 0.99]

#ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
#        transform=ax.transAxes, fontsize = 2, weight = "bold",
#        bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))

#plt.title(f"{run.boilerplate}")


mesh = plot_fq3_flipped(roi, roi.dens, plot = ax, vabsmax = 5, positive = True, downsample = 5)

buffer = 5
yextent = 300 
#plt.xlim(buffer, run.xlim - buffer)
#plt.ylim(0, run.ylim)
#rect_patch = plt.Rectangle((rect[0], rect[2]), rect[1] - rect[0], rect[3] - rect[2], 
#                           linewidth=2, edgecolor='red', facecolor='none', linestyle=':')
#ax.add_patch(rect_patch)
#cbar = fig.colorbar(mesh, ax=ax)
#cbar.set_label("$n / n_0$", rotation=270)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
print(run.steps)

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.patches import ConnectionPatch

In [ ]:
%autoreload

# SETTINGS

lazy = 10
contour_cheat = 10
counter = 0
buffer = 10

flh1 = None
flh2 = None
flh3 = None

fig = plt.figure(figsize=(36, 12))
gs = GridSpec(2, 2, width_ratios=[1.5, 3], height_ratios=[1, 1], wspace=0.3, hspace=0.3)

for rel_ts in range(5):
    title = f"$\Omega t = $ {round(rel_ts * run.gyrofreq):05d}"
    ez_rep = np.nan

    def get_crec(width, height, center = [run.xlim / 2, run.ylim / 2]):
        x1 = center[0] - width / 2
        x2 = center[0] + width / 2
        y1 = center[1] - height / 2
        y2 = center[1] + height / 2
        return [x1, x2, y1, y2]


    rects = [[run.xlim / 2 - 200, run.xlim / 2 + 200, 1300, 1900], get_crec(240, 480)]
    colors = ['yellow', 'red']
    titles = ['Zoom 2', 'Zoom 1']

    def plot_main(ax):
        global flh1
        roi = fq_cut(d.fields, rel_ts, [0 + buffer, run.xlim - buffer, 0, run.ylim / 2])
        ez_rep = roi.ez.mean()
        #ez_rep = roi.ez[0, 0]
        print(ez_rep.compute().item())

        im = plot_fq3_flipped(roi, roi.dens, plot = ax, vabsmax = 5, positive = True, downsample = lazy)

        if (flh1 is None):
            flh1 = FieldLineHelper(run, 10, contour_ds = contour_cheat)
        #flh1.plot_field_lines(roi, ez_rep, plot = ax, arrow_spacing = -1, linewidth = 1, transpose = True) #1000

        ax.set_ylim(buffer, run.xlim - buffer)
        ax.set_title(title)
        #ax.set_title("Main Image")
        return im

    def plot_zoom(ax, rect, color, title, flh_index):
        global flh2, flh3
        curr_flh = [flh2, flh3][flh_index]

        [y1, y2, x1, x2] = rect
        roi = fq_cut(d.fields, rel_ts, rect)
        plot_fq3_flipped(roi, roi.dens, plot = ax, vabsmax = 5, positive = True, downsample = lazy)

        if (curr_flh is None):
            if flh_index == 0:
                flh2 = FieldLineHelper(run, 8, contour_ds = contour_cheat)
            else:
                flh3 = FieldLineHelper(run, 8, contour_ds = contour_cheat)  
        else:
            2 + 2
            print("trying")
            curr_flh.plot_field_lines(roi, ez_rep, plot = ax, arrow_spacing = -1, transpose = True)

        #ax.set_xlim(y1, y2)
        #ax.set_ylim(x1, x2)
        #ax.set_title(title)
        # Show axis ticks and labels
        ax.tick_params(axis='both', which='both', direction='in', length=4)
        ax.set_xlabel("$y$ [$c/\\omega_{p}$]")
        ax.set_ylabel("$x$ [$c/\\omega_{p}$]")
        rect_patch = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor=color, facecolor='none', lw=1.5)
        return rect_patch

    def connect_rect(fig, ax_main, ax_zoom, rect, color):
        [x1, x2, y1, y2] = rect
        con1 = ConnectionPatch(xyA=(y1, x1), coordsA=ax_main.transData,
                            xyB=(0, 1), coordsB=ax_zoom.transAxes,
                            color=color)
        con2 = ConnectionPatch(xyA=(y2, x1), coordsA=ax_main.transData,
                            xyB=(1, 1), coordsB=ax_zoom.transAxes,
                            color=color)
        fig.add_artist(con1)
        fig.add_artist(con2)

    #plt.suptitle(title)

    ax_main = fig.add_subplot(gs[0, :])
    im = plot_main(ax_main)

    zoom_axes = []
    for i, (rect, color, title) in enumerate(zip(rects, colors, titles)):
        ax_zoom = fig.add_subplot(gs[1, i])
        rect_patch = plot_zoom(ax_zoom, rect, color, title, i)
        ax_main.add_patch(rect_patch)
        connect_rect(fig, ax_main, ax_zoom, rect, color)
        zoom_axes.append(ax_zoom)

    #save_polished_fig(name = "bigrun_zooms", dpi = 400)
    save_movie_frame(counter, dpi = 200)

    plt.tight_layout()
    plt.show()
    #plt.close(fig)

    print(f"Done with frame {counter}")

    counter += 1